# Insurance data analysis

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
df = pd.read_csv('/kaggle/input/datasets/mirichoi0218/insurance/insurance.csv')
df.head()

## Basic checks

In [ ]:
df.shape

In [ ]:
df.info()

In [ ]:
df.describe(include='all')

In [ ]:
df.isnull().sum()

In [ ]:
# Fill missing values if there are any
numeric_columns = df.select_dtypes(include=np.number).columns
df[numeric_columns] = df[numeric_columns].fillna(df[numeric_columns].median())

categorical_columns = df.select_dtypes(include='object').columns
for column in categorical_columns:
    df[column] = df[column].fillna(df[column].mode()[0])

In [ ]:
df.duplicated().sum()

In [ ]:
# Remove duplicate rows
df = df.drop_duplicates().reset_index(drop=True)
df.shape

## Outliers

In [ ]:
df[['age', 'bmi', 'children', 'charges']].plot(
    kind='box',
    subplots=True,
    layout=(2, 2)
)
plt.tight_layout()
plt.show()

In [ ]:
# Cap extreme values in BMI and charges using IQR
for column in ['bmi', 'charges']:
    Q1 = df[column].quantile(0.25)
    Q3 = df[column].quantile(0.75)
    IQR = Q3 - Q1

    lower_limit = Q1 - 1.5 * IQR
    upper_limit = Q3 + 1.5 * IQR

    df[column] = df[column].clip(lower_limit, upper_limit)

In [ ]:
df[['bmi', 'charges']].plot(kind='box', subplots=True, layout=(1, 2))
plt.tight_layout()
plt.show()

## EDA

In [ ]:
sns.histplot(df['charges'], bins=30, kde=True)
plt.title('Distribution of charges')
plt.xlabel('Charges')
plt.show()

In [ ]:
sns.boxplot(data=df, x='smoker', y='charges')
plt.title('Charges for smokers and non-smokers')
plt.show()

In [ ]:
sns.heatmap(
    df[['age', 'bmi', 'children', 'charges']].corr(),
    annot=True,
    cmap='coolwarm'
)
plt.title('Correlation heatmap')
plt.show()

In [ ]:
sns.barplot(data=df, x='region', y='charges', errorbar=None)
plt.title('Average charges by region')
plt.xticks(rotation=20)
plt.show()

In [ ]:
sns.scatterplot(data=df, x='age', y='charges', hue='smoker')
plt.title('Age and charges')
plt.show()

## New features

In [ ]:
# Group BMI values
df['bmi_category'] = pd.cut(
    df['bmi'],
    bins=[0, 18.5, 25, 30, np.inf],
    labels=['underweight', 'normal', 'overweight', 'obese']
)

In [ ]:
# Group ages
df['age_group'] = pd.cut(
    df['age'],
    bins=[0, 29, 44, 59, np.inf],
    labels=['18-29', '30-44', '45-59', '60+']
)

In [ ]:
# Combined risk feature
df['high_risk_smoker'] = ((df['smoker'] == 'yes') & (df['bmi'] >= 30)).astype(int)

df.head()

## What I found

In [ ]:
smoker_charges = df.groupby('smoker')['charges'].mean()
region_charges = df.groupby('region')['charges'].mean().sort_values(ascending=False)
bmi_charges = df.groupby('bmi_category', observed=False)['charges'].mean()

print('1. Average charges are much higher for smokers than non-smokers.')
print('   Smoker:', round(smoker_charges['yes'], 2))
print('   Non-smoker:', round(smoker_charges['no'], 2))

print('2. Charges generally increase as age increases.')
print('   Correlation:', round(df['age'].corr(df['charges']), 2))

print('3. Region with the highest average charges:', region_charges.index[0])
print('   Average:', round(region_charges.iloc[0], 2))

print('4. BMI group with the highest average charges:', bmi_charges.idxmax())
print('   Average:', round(bmi_charges.max(), 2))

print('5. Smoking has a stronger relationship with charges than region or number of children.')

- Smokers have much higher medical charges than non-smokers.
- Charges tend to rise with age.
- Higher BMI groups generally have higher average charges.
- Average charges change across regions, but the difference is smaller than the smoker difference.
- The BMI group, age group and high-risk smoker columns may be useful for a future model.

In [ ]:
df.to_csv('insurance_cleaned.csv', index=False)